# SHS generation + analysis

In [ ]:
# Example: two hairpins joined by a linker. Stem 1 pairs with interaction 1.0,
# stem 2 with 0.5, and linker base 25 forms a base triple with pair (31, 47).
# watson_crick_cov is the one approach where every estimator applies.

# Sequence & structure
RNA_SEQ   = "GAGCGGAUCUUCGAAGAUCUGCAACAUAGCCCUGGAGAAACUAUCUAGGACU"
STRUCTURE = "..(((((((......)))))))........((((((.......))))))..."  # used if INTERACTIONS is None
INTERACTIONS = ("[[2, 21, 1.0], [3, 20, 1.0], [4, 19, 1.0], [5, 18, 1.0], [6, 17, 1.0], [7, 16, 1.0], [8, 15, 1.0], "
                "[30, 48, 0.5], [31, 47, 0.5], [32, 46, 0.5], [33, 45, 0.5], [34, 44, 0.5], [35, 43, 0.5], "
                "[25, 31, 0.5]]")
MUTATION_RATES = None                    # optional per-position list; overrides the two rates below
STRUCTURE_PREDICTOR = "rnafold"          # rnafold, spotrna, rnaformer; used if both are None
PDB_ID = None                            # optional identifier for structure prediction

# MSA size / reproducibility
N = 2000
SEED = 42

# Mutation parameters
MUTATION_RATE_UNPAIRED = 0.15
MUTATION_RATE_PAIRED   = 0.25
PAIR_MUTATION_APPROACH = "watson_crick_cov"  # covariance | watson_crick | watson_crick_cov | original | none

# Replacement base of a mutated pair member (watson_crick, watson_crick_cov, original multiplets)
PAIR_WC_PROB     = 0.8
PAIR_WOBBLE_PROB = 0.3
# Pair mutation of basic pairs (original only)
WOBBLE_PROB = 0.1

# insertions / deletions - paired
STEM_SINGLE_INSERTION_PROB  = 0.03
STEM_LONG_INSERTION_PROB    = 0.02
STEM_SINGLE_DELETION_PROB   = 0.05
STEM_PAIR_DELETION_PROB     = 0.1
STEM_MAX_INSERTION_FRACTION = 0.1

# insertions / deletions - unpaired
LOOP_SINGLE_INSERTION_PROB  = 0.08
LOOP_SINGLE_DELETION_PROB   = 0.1
LOOP_LONG_INSERTION_PROB    = 0.04
LOOP_LONG_DELETION_PROB     = 0.05
LOOP_MAX_INSERTION_FRACTION = 0.15
LOOP_MAX_DELETION_FRACTION  = 0.1

## Setup

In [ ]:
import importlib
import sys
from pathlib import Path

%matplotlib inline

_roots = (Path.cwd(), *Path.cwd().parents)
REPO_ROOT = next((p for p in _roots if (p / "SHS-Generator-RNA" / "shs_generator.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the SyntheticEvolution repository root.")
SHS_DIR = (REPO_ROOT / "SHS-Generator-RNA").resolve()
ANALYZER_DIR = (REPO_ROOT / "analysis" / "SHS-Analyzer-RNA").resolve()
for module_dir in (SHS_DIR, ANALYZER_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import pair_map
import generator_config
import json_generator
import shs_generator
import shs_analyzer
importlib.reload(pair_map)
importlib.reload(generator_config)
importlib.reload(json_generator)
importlib.reload(shs_generator)
importlib.reload(shs_analyzer)

print("SHS-Generator dir:", SHS_DIR)
print("SHS-Analyzer dir:", ANALYZER_DIR)

## Generate the SHS

In [ ]:
if INTERACTIONS is not None:
    structure_input = shs_generator.parse_json(INTERACTIONS) if isinstance(INTERACTIONS, str) else INTERACTIONS
elif STRUCTURE:
    structure_input = STRUCTURE
else:
    import structure_predictor
    structure_input = structure_predictor.predict(STRUCTURE_PREDICTOR, RNA_SEQ, PDB_ID)

per_position_rates = (
    shs_generator.parse_json(MUTATION_RATES)
    if isinstance(MUTATION_RATES, str) else MUTATION_RATES
)
pair_map = generator_config.build_pair_map(
    RNA_SEQ, structure_input,
    mutation_rate_paired=MUTATION_RATE_PAIRED,
    mutation_rate_unpaired=MUTATION_RATE_UNPAIRED,
    mutation_rates=per_position_rates,
)

# Only global algorithm settings belong in MutationParameters.
parameters = generator_config.MutationParameters(
    N=N,
    pair_mutation_approach=PAIR_MUTATION_APPROACH,
    stem_single_insertion_prob=STEM_SINGLE_INSERTION_PROB,
    stem_long_insertion_prob=STEM_LONG_INSERTION_PROB,
    stem_single_deletion_prob=STEM_SINGLE_DELETION_PROB,
    stem_pair_deletion_prob=STEM_PAIR_DELETION_PROB,
    stem_max_insertion_fraction=STEM_MAX_INSERTION_FRACTION,
    loop_single_insertion_prob=LOOP_SINGLE_INSERTION_PROB,
    loop_single_deletion_prob=LOOP_SINGLE_DELETION_PROB,
    loop_long_insertion_prob=LOOP_LONG_INSERTION_PROB,
    loop_long_deletion_prob=LOOP_LONG_DELETION_PROB,
    loop_max_insertion_fraction=LOOP_MAX_INSERTION_FRACTION,
    loop_max_deletion_fraction=LOOP_MAX_DELETION_FRACTION,
    wobble_prob=WOBBLE_PROB,
    pair_wc_prob=PAIR_WC_PROB,
    pair_wobble_prob=PAIR_WOBBLE_PROB,
)
generator = shs_generator.MsaGenerator(parameters, seed=SEED)
msa = generator.generate(RNA_SEQ, pair_map)

# JSON construction is separate from generation. This notebook does not save it.
generated_json = json_generator.build_input_json(
    RNA_SEQ, msa, name="notebook_rna", model_seeds=[1],
)
print("Generated", len(msa), "MSA rows for", len(RNA_SEQ), "RNA positions")


## Analyze & render

In [ ]:
import json
import numpy as np

data = shs_analyzer.MsaData.from_text(json.dumps(generated_json), pairs=pair_map)
# To analyze an exported MSA instead: data = shs_analyzer.MsaData.from_path("<af3_input.json or .fasta>", pair_map)
feat = shs_analyzer.Features(data)

print(f"Query length: {len(data.seq)}  "
   f"|  MSA rows: {data.aligned.shape[0] + 1}  "
   f"|  covariance shape: {feat.covariance.shape}")

print(data.seq)
for row in data.raw[:10]:
    print("".join(row))

# Expected covariance. With covariance and watson_crick_cov, the later base b of a pair (a, b)
# mutates more often if a mutated: cov(a, b) = I * m_b * (1 - m_a) / (k + 1), where k is the
# number of earlier partners of b (1 for a basic pair). none and watson_crick mutate pair
# members independently (0). original has no closed form and is shown as 0 too.
rates = np.diag(pair_map._pairs_mat)
expectation = np.diag(rates)
if PAIR_MUTATION_APPROACH in ("covariance", "watson_crick_cov"):
    for a, b in pair_map.unique_pairs:
        k = sum(j < b for j in pair_map.partners(b))
        expectation[a, b] = expectation[b, a] = pair_map.interaction(a, b) * rates[b] * (1 - rates[a]) / (k + 1)
error = feat.covariance - expectation
vmin = min(np.nanmin(expectation), np.nanmin(feat.covariance))
vmax = max(np.nanmax(expectation), np.nanmax(feat.covariance))
shs_analyzer.plot_covariance(data.seq, expectation, title="Expected covariance and input mutation rates", vmin=vmin, vmax=vmax)
shs_analyzer.plot_covariance(data.seq, feat.covariance, vmin=vmin, vmax=vmax)
# Own symmetric scale, so that systematic errors stand out from the sampling noise.
e = np.nanmax(np.abs(error))
_ = shs_analyzer.plot_covariance(data.seq, error, title="Error (recovered - expected)", vmin=-e, vmax=e)

## Covariance per pair

In [ ]:
_ = shs_analyzer.plot_covariance_classification(data.seq, feat.covariance, pairs=data.pairs)

## Deletion and Insertion analysis

In [ ]:
_ = shs_analyzer.plot_deletions_and_insertion(feat.col_deletion_rate, feat.col_insertion_mean, pairs=data.pairs)

In [ ]:
# Alternative: split panels. Deletion rate per position (top) and insertion rate per gap
# (bottom, share of rows with any insertion) on a shared x-axis.
_ = shs_analyzer.plot_indels(feat.col_deletion_rate, feat.col_insertion_rate, pairs=data.pairs)

## Mutation-rate comparison

In [ ]:
# PairMap stores the effective rates, including scalar defaults and overrides.
input_rates = np.array([pair_map.mutation_rate(i) for i in range(len(data.seq))])
output_rates = feat.col_mut_rate
fig, metrics = shs_analyzer.plot_mutation_rate_comparison(input_rates, output_rates)
print(f"MAE: {metrics['mae']:.4f} | RMSE: {metrics['rmse']:.4f} | Pearson r: {metrics['pearson_r']:.4f}")


## Recovered parameters

In [ ]:
import pandas as pd

# Show full cell contents so the `detail` strings aren't truncated.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Known generator input per recovered parameter name (the ground truth each
# estimator tries to reproduce). Extend this as you add estimators.
# pair_wc_prob and pair_wobble_prob are only recovered for watson_crick and
# watson_crick_cov, interaction_strength only for covariance and watson_crick_cov.
from dataclasses import asdict

L = len(RNA_SEQ)
KNOWN_INPUTS = {
    **asdict(parameters),
    "n_sequences":            N,
    "mutation_rate_unpaired": MUTATION_RATE_UNPAIRED,
    "mutation_rate_paired":   MUTATION_RATE_PAIRED,
    # Long insertions have at most max(int(L * fraction), 2) bases.
    "loop_max_insertion_fraction": max(int(L * LOOP_MAX_INSERTION_FRACTION), 2) / L,
    "stem_max_insertion_fraction": max(int(L * STEM_MAX_INSERTION_FRACTION), 2) / L,
    "interaction_strength": (np.mean([pair_map.interaction(a, b) for a, b in pair_map.basic_pairs])
                             if pair_map.basic_pairs else None),
}

report = shs_analyzer.analyze_data(data)


def _fmt_detail(detail):
    # Turn {'unpaired_cols': 14, ...} into a readable "unpaired_cols=14 · ..." line.
    return " · ".join(f"{k}={v}" for k, v in detail.items())


rows = []
for name, est in report.items():
    known = KNOWN_INPUTS.get(name)
    # est.value != est.value is True only for NaN -> no error there.
    error = abs(est.value - known) if (known is not None and est.value == est.value) else None
    rows.append({
        "parameter": name,
        "known":     known,
        "recovered": round(est.value, 4),
        "abs_error": round(error, 4) if error is not None else None,
        "detail":    _fmt_detail(est.detail),
    })

pd.DataFrame(rows).set_index("parameter")